# REAL Video Enhancer Colab Upscale

Colab 上で RVE backend を使い、動画を 2x アップスケールします。

事前に Colab の `ランタイム > ランタイムのタイプを変更` で GPU を選択してください。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print(sys.version)
print(subprocess.check_output(['nvidia-smi'], text=True) if Path('/usr/bin/nvidia-smi').exists() else 'nvidia-smi not found')

## 1. リポジトリを用意

`REPO_URL` は GitHub に push した後、自分のリポジトリ URL に変更してください。Colab に直接ファイルをアップロードして使う場合は、このセルを実行せず、`WORKDIR` をアップロード先に合わせます。

In [ ]:
REPO_URL = 'https://github.com/YOUR_USER/REAL-Video-Enhancer.git'
BRANCH = 'main'
WORKDIR = Path('/content/REAL-Video-Enhancer')

if not WORKDIR.exists():
    if 'YOUR_USER' in REPO_URL:
        raise ValueError('REPO_URL を自分の GitHub リポジトリ URL に変更してください。')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(WORKDIR)], check=True)

os.chdir(WORKDIR)
print('WORKDIR =', Path.cwd())

## 2. ffmpeg と Python 依存関係をインストール

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg git
!python -m pip install -U -q pip
!python -m pip install -q \
  sympy tqdm typing_extensions packaging requests opencv-python-headless \
  pypresence mpmath pillow numpy==2.2.2 scenedetect \
  torch==2.7.0 torchvision==0.22.0 einops safetensors

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. 入出力フォルダとモデルを用意

In [ ]:
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)

MODEL_PATH = Path('models/up2x-latest-conservative.pth')
MODEL_URL = 'https://github.com/TNTwise/real-video-enhancer-models/releases/download/models/up2x-latest-conservative.pth'

if not MODEL_PATH.exists():
    subprocess.run(['wget', '-q', '--show-progress', '-O', str(MODEL_PATH), MODEL_URL], check=True)

print(MODEL_PATH, MODEL_PATH.stat().st_size, 'bytes')

## 4A. 動画をアップロードして使う場合

アップロード後、最初のファイルを `input/video.mp4` として保存します。大きい動画では Google Drive の利用を推奨します。

In [ ]:
from google.colab import files
uploaded = files.upload()

if uploaded:
    name = next(iter(uploaded.keys()))
    src = Path(name)
    dst = Path('input/video.mp4')
    src.rename(dst)
    print('saved:', dst)

## 4B. Google Drive を使う場合

Drive 上の動画を使う場合はこちらを使います。`DRIVE_INPUT` を実ファイルに変更してください。

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_INPUT = Path('/content/drive/MyDrive/video.mp4')
# INPUT_PATH = DRIVE_INPUT
# OUTPUT_PATH = Path('/content/drive/MyDrive/video_2x.mp4')

INPUT_PATH = Path('input/video.mp4')
OUTPUT_PATH = Path('output/video_2x.mp4')

print('input:', INPUT_PATH)
print('output:', OUTPUT_PATH)

## 5. 動画情報を確認

In [ ]:
assert INPUT_PATH.exists(), f'入力動画がありません: {INPUT_PATH}'
!python backend/rve-backend.py --ffmpeg_path /usr/bin/ffmpeg --print_video_info "{INPUT_PATH}"

## 6. アップスケール実行

GPU ランタイムでは `--device cuda` を使います。CPU しかない場合は `--device cpu` に変更できますが、非常に遅くなります。

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cmd = [
    sys.executable,
    'backend/rve-backend.py',
    '-i', str(INPUT_PATH),
    '-o', str(OUTPUT_PATH),
    '--ffmpeg_path', '/usr/bin/ffmpeg',
    '-b', 'pytorch',
    '--device', DEVICE,
    '--upscale_model', str(MODEL_PATH),
    '--crf', '18',
    '--video_encoder_preset', 'libx264',
    '--video_pixel_format', 'yuv420p',
    '--overwrite',
]

print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 7. 出力をダウンロード

In [ ]:
assert OUTPUT_PATH.exists(), f'出力ファイルがありません: {OUTPUT_PATH}'
files.download(str(OUTPUT_PATH))